# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-6-luna"   # 호출마다 reasoning_effort="none" 을 함께 넣는다 (교재 5.3)
print("client ready:", MODEL)

client ready: gpt-6-luna


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [19]:
resp = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "고양이를 한 문장으로 소개해줘."},
    ],
    temperature=0.5,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

고양이는 호기심 많고 독립적이며, 부드러운 털과 귀여운 몸짓으로 사랑받는 반려동물입니다.
usage: CompletionUsage(completion_tokens=37, prompt_tokens=34, total_tokens=71, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [9]:
stream = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
    messages=[{"role": "user", "content": "LLM 을 13줄로 설명해줘."}],
    stream=True,
)
print(stream)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

1. LLM은 ‘대규모 언어 모델’(Large Language Model)의 약자입니다.
2. 많은 텍스트를 학습해 언어의 패턴을 익힙니다.
3. 질문에 답하고 글을 쓰며 요약하거나 번역할 수 있습니다.
4. 문맥을 바탕으로 다음에 올 단어를 예측해 답변을 만듭니다.
5. 대화형 AI와 글쓰기 도구 등에 널리 쓰입니다.
6. 사람처럼 문장을 이해하고 표현하는 듯한 결과를 냅니다.
7. 하지만 사람처럼 경험하거나 의식을 가진 것은 아닙니다.
8. 학습 데이터에 포함된 편견이나 오류를 답변에 반영할 수 있습니다.
9. 그럴듯하지만 틀린 정보를 만들어낼 때도 있습니다.
10. 최신 정보나 전문 지식은 별도로 확인하는 것이 좋습니다.
11. 질문이 구체적일수록 더 유용한 답을 얻기 쉽습니다.
12. 개인정보나 민감한 정보는 입력하지 않는 편이 안전합니다.
13. LLM은 사람의 판단을 돕는 도구로 활용하는 것이 바람직합니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [10]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    60
0    40
Name: count, dtype: int64


,clean_text,label
0,재구매 의사 100 강추합니다,1
1,사이즈도 딱 맞고 마감이 깔끔해요,1
2,가성비 최고입니다 또 살게요,1
3,색감이 사진이랑 똑같아서 만족합니다,1
4,배송이 일주일이나 걸렸습니다 별로,0


In [11]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

라벨: 1 / 토큰: 83


In [20]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8591


### sklearn(실습 2) 과 비교 — 정확도

In [21]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])
print(f"LLM 정확도: {acc:.3f}")
print(classification_report(sample["label"], sample["pred"], digits=3))
# TODO: 실습 2 sklearn 정확도와 한 표로 비교 (정확도 / 비용 / 속도 / 일관성)

LLM 정확도: 1.000
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        40
           1      1.000     1.000     1.000        60

    accuracy                          1.000       100
   macro avg      1.000     1.000     1.000       100
weighted avg      1.000     1.000     1.000       100



## 3) Task 3 — 비용 측정 + 1만 건 추산

In [22]:
# gpt-6-luna 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.10/M, 출력 0.50/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.10 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens / len(sample)
cost_100   = tokens * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")
# TODO: ML vs LLM — '언제 무엇을 쓸지' 한 줄 결론을 적는다

평균 토큰/건: 85.9
100건 비용: $0.0009
1만 건 추산: $0.09


## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*